<a href="https://colab.research.google.com/github/SilviaPorciuleanu/west-narratives-party-manifestos/blob/main/05_Final_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

In [ ]:
from pathlib import Path
import hashlib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/West_Narratives"  # change if your folder has another name

Mounted at /content/drive


In [ ]:
# Path to your coded output file
INPUT_FILE = f"{PROJECT_DIR}/manifesos_west_dictionary_based_unique_15102026_cleaned.csv"

# Import coded data
df = pd.read_csv(INPUT_FILE)


In [ ]:
df.head()

,docname,from,to,pre,keyword,post,lenght,text,referent_category,api_error
0,80510_201305,8,8,Strict compliance with the polluter pays princ...,We are IMPLEMENTING A TRANSPARENT AND EFFECTIV...,The efficient management of financial resource...,35,Strict compliance with the polluter pays princ...,"EU, NATO and European structures",NaN
1,80510_201305,11,11,We will achieve this by: increasing funding fo...,Creating programmes for the renovation of gyms...,Guidelines for combining sports training and g...,24,We will achieve this by: increasing funding fo...,"EU, NATO and European structures",NaN
2,80510_201410,2,2,"EU Funds - a tool and incentive for reforms, g...",Restoring the confidence of the European insti...,Restoring the payments under the suspended ope...,7,"EU Funds - a tool and incentive for reforms, g...","EU, NATO and European structures",NaN
3,80510_201410,11,11,Opening new markets for Bulgarian businesses a...,Supporting the European perspective of the Wes...,The Bulgarian Army - a reliable guarantor of t...,10,Opening new markets for Bulgarian businesses a...,Western signifier,NaN
4,80510_201410,21,21,Opening new markets for Bulgarian businesses a...,Supporting the European perspective of the Wes...,The Bulgarian Army - a reliable guarantor of t...,10,Opening new markets for Bulgarian businesses a...,Western signifier,NaN


In [ ]:
# Manifesto-level variables file
vars_file = f"{PROJECT_DIR}/Manifestos_variables_120526.csv"

df_vars = pd.read_csv(vars_file)


In [ ]:
df_vars.head()

,manifesto_id,party,date,language,country_code,year,month,partyname,parfam
0,80220_199412,80220,199412,english,BGR,1994,12,Bulgarian Socialist Party,30.0
1,80410_199412,80410,199412,english,BGR,1994,12,Union of Democratic Forces,40.0
2,80811_199412,80811,199412,english,BGR,1994,12,People’s Union,70.0
3,80951_199412,80951,199412,english,BGR,1994,12,Movement for Rights and Freedoms,90.0
4,80221_200907,80221,200907,bulgarian,BGR,2009,7,Coalition for Bulgaria,30.0


In [ ]:
df_coded_with_vars = df.merge(
    df_vars,
    how="left",
    left_on="docname",
    right_on="manifesto_id",
    validate="many_to_one",
    indicator=True
)

print("Original sentences:", len(df))
print("After merge:", len(df_coded_with_vars))
print(df_coded_with_vars["_merge"].value_counts())

if (df_coded_with_vars["_merge"] != "both").any():
    raise ValueError("Some sentences did not match a manifesto.")

df_coded_with_vars = df_coded_with_vars.drop(columns="_merge")

Original sentences: 11535
After merge: 11535
_merge
both          11535
left_only         0
right_only        0
Name: count, dtype: int64


In [ ]:
df_coded_with_vars.head()

,docname,from,to,pre,keyword,post,lenght,text,referent_category,api_error,manifesto_id,party,date,language,country_code,year,month,partyname,parfam
0,80510_201305,8,8,Strict compliance with the polluter pays princ...,We are IMPLEMENTING A TRANSPARENT AND EFFECTIV...,The efficient management of financial resource...,35,Strict compliance with the polluter pays princ...,"EU, NATO and European structures",NaN,80510_201305,80510,201305,bulgarian,BGR,2013,5,Citizens for European Development of Bulgaria,50.0
1,80510_201305,11,11,We will achieve this by: increasing funding fo...,Creating programmes for the renovation of gyms...,Guidelines for combining sports training and g...,24,We will achieve this by: increasing funding fo...,"EU, NATO and European structures",NaN,80510_201305,80510,201305,bulgarian,BGR,2013,5,Citizens for European Development of Bulgaria,50.0
2,80510_201410,2,2,"EU Funds - a tool and incentive for reforms, g...",Restoring the confidence of the European insti...,Restoring the payments under the suspended ope...,7,"EU Funds - a tool and incentive for reforms, g...","EU, NATO and European structures",NaN,80510_201410,80510,201410,bulgarian,BGR,2014,10,Citizens for European Development of Bulgaria,50.0
3,80510_201410,11,11,Opening new markets for Bulgarian businesses a...,Supporting the European perspective of the Wes...,The Bulgarian Army - a reliable guarantor of t...,10,Opening new markets for Bulgarian businesses a...,Western signifier,NaN,80510_201410,80510,201410,bulgarian,BGR,2014,10,Citizens for European Development of Bulgaria,50.0
4,80510_201410,21,21,Opening new markets for Bulgarian businesses a...,Supporting the European perspective of the Wes...,The Bulgarian Army - a reliable guarantor of t...,10,Opening new markets for Bulgarian businesses a...,Western signifier,NaN,80510_201410,80510,201410,bulgarian,BGR,2014,10,Citizens for European Development of Bulgaria,50.0


In [ ]:
country_map = {
    "BGR": "Bulgaria",
    "CZE": "Czech Republic",
    "EST": "Estonia",
    "HRV": "Croatia",
    "HUN": "Hungary",
    "LTU": "Lithuania",
    "LVA": "Latvia",
    "POL": "Poland",
    "ROU": "Romania",
    "SVK": "Slovakia",
    "SVN": "Slovenia"
}

df_coded_with_vars["country_name"] = df_coded_with_vars["country_code"].map(country_map)

df_coded_with_vars[["country_code", "country_name"]].drop_duplicates().sort_values("country_code")

,country_code,country_name
0,BGR,Bulgaria
6,CZE,Czech Republic
54,EST,Estonia
102,HUN,Hungary
125,LTU,Lithuania
122,LVA,Latvia
247,POL,Poland
317,ROU,Romania
447,SVK,Slovakia
604,SVN,Slovenia


In [ ]:
print(
    df_coded_with_vars["referent_category"]
    .value_counts(dropna=False)
)

print(
    "Missing country names:",
    df_coded_with_vars["country_name"].isna().sum()
)

referent_category
EU, NATO and European structures    5713
Western signifier                   3513
Western countries                   1682
West                                 627
Name: count, dtype: int64
Missing country names: 0


In [ ]:
TEXT_COLUMN = "keyword"              # target sentence
PREVIOUS_COLUMN = "pre"              # sentence before
NEXT_COLUMN = "post"                 # sentence after
REFERENT_COLUMN = "referent_category" # actor/referent already coded
COUNTRY_COLUMN = "country_name"       # country name

In [ ]:
import json
import time
from tqdm import tqdm
from openai import OpenAI
from google.colab import userdata

In [ ]:
# API key
api_key = userdata.get("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets.")

client = OpenAI(api_key=api_key)

In [ ]:
schema_self_other_valence = {
    "type": "object",
    "properties": {
        "self_other": {
            "type": "integer",
            "enum": [-1, 1, 2]
        },
        "valence": {
            "type": "integer",
            "enum": [-1, 1, 2]
        }
    },
    "required": ["self_other", "valence"],
    "additionalProperties": False
}

In [ ]:
def build_prompt_self_other_valence(
    target_sentence,
    country=None,
    referent_category=None,
    previous_sentence=None,
    next_sentence=None
):

    return f"""
You are coding sentences from party manifestos in Central and Eastern Europe. Your task is to code only two variables: self_other and valence.

**Inputs:**
Country of manifesto: {country}
West-related actor/referent category to code: {referent_category}
Previous sentence: {previous_sentence}
TARGET SENTENCE: {target_sentence}
Next sentence: {next_sentence}

**Important unit rule:**
• Code only the TARGET SENTENCE.
• Use the previous and next sentences to better understand the context, clarify pronouns, implicit referents, incomplete phrases, or ambiguous terms.
• Code the provided West-related actor/referent, not the sentence in general.
• If a sentence contains multiple geopolitical actors, do not average them.
• Code the two variables from the perspective of the manifesto country.
• Code self_other and valence independently.

**Axis 1: self_other**
Code how the target sentence positions the party's country/national community in relation to the provided West-related actor/referent.

**Allowed values:**

1 = self

**Definition:** The sentence presents the country as Western, European, or as part of, belonging to, acting within or on behalf of the West/Europe/EU/NATO/Western countries.

**Clarification:** The sentence does not need to explicitly say “we are European” or “we are members.” If it suggests equal relations with Western countries or full membership in the West or Europe, it is self.

**Negative clarification:** Positive does not mean self. For instance, the West/Europe or Western countries used as a comparison or model, the classification would not be self, it would be other.

**Positive example:** "The change of regime has confirmed the historical place of the Hungarian nation in Europe: we are not a ferry country between East and West, but unquestionably the West, the European Union and an integral part of the transatlantic community."

**Negative example coded as other:** "Internationally, Slovenia must continue to consolidate its role as a sovereign and equal member of the European Union and the Atlantic Alliance."

-1 = other

**Definition:** The sentence presents the West/Europe/EU/NATO/Western countries as external, or suggests that their country is not fully Western/European or not yet integrated into the West/Europe/EU/NATO.

**Clarification:** If the country needs to catch up or gain influence to be more similar to Western/European actors, it is classified as other.

**Negative clarification:** For instance, taking a position on behalf of Western/European countries or organisations or discussing common policy issues from an equal position, it is classified as self, not as other.

**Positive example:** "Through greater involvement in the European integration process and the development of transatlantic ties, we will fundamentally strengthen the international position and security of the Czech Republic."

**Negative example coded as self:** "We actively support the development of the European Union's common foreign and security policy."

2 = not applicable

**Definition:** There is no broader identity or relational meaning toward the West/Europe/Western countries/EU/NATO.

**Clarification:** This includes simple descriptive, geographic, and biographic mentions. For instance, geographic locations would be classified as not applicable.

**Negative clarification:** If the mention of a Western actor is meaningful, for instance, suggesting a higher or lower status, identity or membership, then it is not classified as not applicable.

**Positive example:** "In order to maintain a living link with the motherland, a new institutional system will be established to provide administrative, social, cultural, legal and contact support to the Hungarian diaspora in the West and to expatriates."

**Negative example coded as self:** "We will do our utmost to ensure that Lithuania develops and maximises all possible foreign policy instruments in its relations with Germany, France, Italy, Spain, the UK, Scandinavia and the Benelux countries, in line with its own positions on issues of principle."

**Axis 2: valence**
Code the evaluative or relational orientation of the target sentence toward the provided West-related actor/referent.

**Allowed values:**

1 = positive

**Definition:** The West/Europe/Western countries/EU/NATO are framed as positive, desirable, beneficial, legitimate, protective, supportive, advanced, useful, worth supporting, worth defending, worth improving, or part of the solution.

**Clarification:** Negative aspects in the sentence do not automatically mean negative if the West-related actor/referent remains framed positively.

**Negative clarification:** If the West/Europe/Western countries/EU/NATO are presented as negative warning cases, the mention would be classified as negative.

**Positive example:** "Today, in the face of such "societal" problems, it is time to think about how we would like to be, now that we are part of Europe, the West."

**Negative example coded as negative:** "It appears that the parties, referring to Brussels for their every action or thesis, which have ruled so far, do not even respect the principles of their beloved EU."

-1 = negative

**Definition:** The West/Europe/Western countries/EU/NATO are framed as negative, harmful, threatening, unstable, unfair, subordinating, imposing, failed, problematic, or as a negative warning example.

**Clarification:** Do not infer negative valence from the general negativity of the sentence.

**Negative clarification:** If the sentence discusses a negative aspect in the West/Europe/Western countries/EU/NATO but the general position towards the actor is still positive, the classification is positive.

**Positive example:** "Its root cause is that Western civilisation, of which Lithuania is a part, has in its historical development broken away from its classical - ancient and Christian - roots."

**Negative example coded as positive:** "We are also lagging far behind compared to the tendencies of the Western states."

2 = not applicable

**Definition:** There is no evaluative position toward the West/Europe/Western countries/EU/NATO.

**Clarification:** This includes simple descriptive, geographic, and biographic mentions. For instance, geographic locations would be classified as not applicable.

**Negative clarification:** If EU/NATO/Western institutions are presented as sources of security, funding, rights, standards, democracy, solidarity, or other benefits, and an evaluative position is attached to them, they are not classified as not applicable.

**Positive example:** "ENEMIES OF FREEDOM AND OUR RESISTANCE After the criminal German and Soviet occupation, basic values were rejected by the Communists using force and mass repression."

**Negative example coded as positive:** "Promoting an active and targeted policy towards countries in Europe's immediate neighbourhood, particularly in North Africa and the Middle East, to enhance stability and prosperity in these countries, thereby reducing security risks."

Return only valid JSON in this exact format:
{{
  "self_other": 1,
  "valence": 1
}}
"""

In [ ]:
def code_self_other_valence(
    target_sentence,
    country=None,
    referent_category=None,
    previous_sentence=None,
    next_sentence=None,
    model=None,
    max_retries=3
):

    if pd.isna(target_sentence) or str(target_sentence).strip() == "":
        return {
            "self_other": None,
            "valence": None,
            "error": "empty target sentence",
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None
        }

    prompt = build_prompt_self_other_valence(
        target_sentence=str(target_sentence),
        country="" if pd.isna(country) else str(country),
        referent_category="" if pd.isna(referent_category) else str(referent_category),
        previous_sentence="" if pd.isna(previous_sentence) else str(previous_sentence),
        next_sentence="" if pd.isna(next_sentence) else str(next_sentence)
    )

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model,
                input=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                reasoning={
                    "effort": "medium"
                },
                max_output_tokens=2000,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "west_self_other_valence",
                        "strict": True,
                        "schema": schema_self_other_valence
                    }
                }
            )

            # Check response status
            if response.status != "completed":
                raise ValueError(
                    f"Response incomplete: {response.status}"
                )

            # Parse JSON
            result = json.loads(response.output_text)

            # Validate labels
            valid_labels = [-1, 1, 2]

            if (
                result.get("self_other") not in valid_labels
                or result.get("valence") not in valid_labels
            ):
                raise ValueError(f"Invalid labels: {result}")

            # Token usage
            usage = (
                response.usage.model_dump()
                if response.usage is not None
                else {}
            )

            return {
                "self_other": result["self_other"],
                "valence": result["valence"],
                "error": None,
                "input_tokens": usage.get("input_tokens"),
                "output_tokens": usage.get("output_tokens"),
                "total_tokens": usage.get("total_tokens")
            }

        except Exception as e:

            if attempt == max_retries - 1:
                return {
                    "self_other": None,
                    "valence": None,
                    "error": str(e),
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None
                }

            time.sleep(2 ** attempt)

In [ ]:
print("Total observations:", len(df_coded_with_vars))

print("\nReferent distribution:")
print(
    df_coded_with_vars["referent_category"]
    .value_counts(dropna=False)
)

print("\nMissing values:")
print(
    df_coded_with_vars[
        [TEXT_COLUMN, REFERENT_COLUMN, COUNTRY_COLUMN]
    ].isna().sum()
)

print("\nDuplicate manifesto-sentence records:")
print(
    df_coded_with_vars.duplicated(
        subset=["docname", TEXT_COLUMN]
    ).sum()
)

print("\nCountry distribution:")
print(
    df_coded_with_vars["country_name"]
    .value_counts(dropna=False)
)

Total observations: 11535

Referent distribution:
referent_category
EU, NATO and European structures    5713
Western signifier                   3513
Western countries                   1682
West                                 627
Name: count, dtype: int64

Missing values:
keyword              0
referent_category    0
country_name         0
dtype: int64

Duplicate manifesto-sentence records:
279

Country distribution:
country_name
Lithuania         1938
Slovenia          1863
Poland            1741
Hungary           1521
Slovakia          1372
Czech Republic    1040
Bulgaria           714
Romania            689
Estonia            608
Latvia              49
Name: count, dtype: int64


In [ ]:
print("\nCountry distribution:")
print(
    df_coded_with_vars["language"]
    .value_counts(dropna=False)
)


Country distribution:
language
lithuanian    1938
slovenian     1863
polish        1731
hungarian     1521
slovak        1365
czech         1040
romanian       689
bulgarian      678
estonian       608
english         53
latvian         49
Name: count, dtype: int64


In [ ]:
# ============================================================
# 1. SETTINGS
# ============================================================

MODEL = "gpt-6.1-sol"
RUN_TAG = "full_dataset_medium"

TEST_ONLY = False           # First test one sentence
RESUME_EXISTING = True
CHECKPOINT_EVERY = 200
MAX_RETRIES = 1
MAX_CONSECUTIVE_ERRORS = 3

LABELS = [-1, 1, 2]

PROJECT_DIR = Path("/content/drive/MyDrive/West_Narratives")

OUTPUT_FILE = PROJECT_DIR / (
    f"west_self_other_valence_gpt-6_1-sol_{RUN_TAG}.csv"
)

RECOVERY_FILE = PROJECT_DIR / (
    f"west_self_other_valence_gpt-6_1-sol_{RUN_TAG}_recovery.jsonl"
)

# Check that Drive is available
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError(
        "Google Drive is not mounted. Mount Drive before running."
    )

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Folder not found: {PROJECT_DIR}")

# ============================================================
# 2. PREPARE FULL DATASET
# ============================================================

df_run = df_coded_with_vars.reset_index(drop=True).copy()

for col in [PREVIOUS_COLUMN, NEXT_COLUMN, COUNTRY_COLUMN]:
    if col not in df_run.columns:
        df_run[col] = ""

required_columns = [
    TEXT_COLUMN,
    REFERENT_COLUMN
]

for col in required_columns:
    if col not in df_run.columns:
        raise ValueError(f"Missing input column: {col}")

print("Total sentences:", len(df_run))

if len(df_run) <= 200:
    raise ValueError(
        "The input contains 200 or fewer sentences. "
        "Check that df_coded_with_vars is your FULL dataset "
        "rather than the validation sample."
    )

# Store the exact prompt fingerprint for each row
df_run["annotation_prompt_sha256"] = df_run.apply(
    lambda r: hashlib.sha256(
        build_prompt_self_other_valence(
            target_sentence=(
                "" if pd.isna(r[TEXT_COLUMN])
                else str(r[TEXT_COLUMN])
            ),
            country=(
                "" if pd.isna(r[COUNTRY_COLUMN])
                else str(r[COUNTRY_COLUMN])
            ),
            referent_category=(
                "" if pd.isna(r[REFERENT_COLUMN])
                else str(r[REFERENT_COLUMN])
            ),
            previous_sentence=(
                "" if pd.isna(r[PREVIOUS_COLUMN])
                else str(r[PREVIOUS_COLUMN])
            ),
            next_sentence=(
                "" if pd.isna(r[NEXT_COLUMN])
                else str(r[NEXT_COLUMN])
            )
        ).encode("utf-8")
    ).hexdigest(),
    axis=1
)

OUTPUT_COLUMNS = [
    "self_other_api_new",
    "valence_api_new",
    "api_error",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "cached_input_tokens",
    "elapsed_seconds"
]

# ============================================================
# 3. LOAD CHECKPOINT OR START NEW
# ============================================================

if OUTPUT_FILE.exists():

    if not RESUME_EXISTING:
        raise FileExistsError(
            "An existing output file was found. "
            "Use RESUME_EXISTING=True or choose a new RUN_TAG."
        )

    df_model = pd.read_csv(OUTPUT_FILE)

    if len(df_model) != len(df_run):
        raise ValueError(
            "The saved checkpoint and current dataset "
            "have different numbers of rows."
        )

    if "annotation_prompt_sha256" not in df_model.columns:
        raise ValueError(
            "Existing checkpoint has no prompt fingerprints. "
            "Do not overwrite it; use a new RUN_TAG."
        )

    if not df_model["annotation_prompt_sha256"].equals(
        df_run["annotation_prompt_sha256"]
    ):
        raise ValueError(
            "Input data or annotation prompt has changed. "
            "The existing checkpoint has been preserved."
        )

    print("Existing checkpoint verified.")

else:

    df_model = df_run.copy()
    print("Starting a new full-dataset annotation.")

for col in OUTPUT_COLUMNS:
    if col not in df_model.columns:
        df_model[col] = None

df_model["model"] = MODEL
df_model["reasoning_effort"] = "medium"


# ============================================================
# 4. SAFE CHECKPOINT FUNCTION
# ============================================================

def save_full_checkpoint():

    temporary_file = OUTPUT_FILE.with_suffix(".csv.tmp")

    df_model.to_csv(
        temporary_file,
        index=False,
        encoding="utf-8"
    )

    os.replace(temporary_file, OUTPUT_FILE)

    print(f"Full CSV checkpoint saved: {OUTPUT_FILE.name}")


# ============================================================
# 5. RECOVER ANNOTATIONS FROM THE JOURNAL
# ============================================================

recovered = 0

if RECOVERY_FILE.exists():

    print("\nChecking recovery log...")

    with open(RECOVERY_FILE, "r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError:
                raise ValueError(
                    f"Recovery log has a damaged entry at "
                    f"line {line_number}. Preserve the file "
                    "and inspect it before resuming."
                )

            if record.get("model") != MODEL:
                raise ValueError(
                    "Recovery log contains a different model."
                )

            i = int(record["row_index"])

            if i < 0 or i >= len(df_model):
                raise ValueError("Invalid row index in recovery log.")

            if (
                record["prompt_sha256"]
                != df_model.loc[i, "annotation_prompt_sha256"]
            ):
                raise ValueError(
                    f"Recovery prompt mismatch at row {i}. "
                    "Stopping to protect existing annotations."
                )

            # Do not overwrite successfully completed observations
            existing_self = pd.to_numeric(
                df_model.loc[i, "self_other_api_new"],
                errors="coerce"
            )
            existing_valence = pd.to_numeric(
                df_model.loc[i, "valence_api_new"],
                errors="coerce"
            )

            if (
                existing_self in LABELS
                and existing_valence in LABELS
            ):
                continue

            for col in OUTPUT_COLUMNS:
                df_model.loc[i, col] = record.get(col)

            recovered += 1

    print("Annotations recovered from log:", recovered)

else:
    print("No previous recovery log found.")

# ============================================================
# 6. FIND UNFINISHED OBSERVATIONS
# ============================================================

completed = (
    pd.to_numeric(
        df_model["self_other_api_new"],
        errors="coerce"
    ).isin(LABELS)
    &
    pd.to_numeric(
        df_model["valence_api_new"],
        errors="coerce"
    ).isin(LABELS)
)

pending = df_model.index[~completed].tolist()

if TEST_ONLY:
    pending = pending[:1]

print("\n=========================================")
print("GPT-6.1 SOL — FULL DATASET")
print("=========================================")
print("Total observations:", len(df_model))
print("Already completed:", int(completed.sum()))
print("Remaining:", int((~completed).sum()))
print("Requests planned this session:", len(pending))
print("Test mode:", TEST_ONLY)
print("=========================================\n")

# Persist recovered entries and the starting dataset
save_full_checkpoint()


# ============================================================
# 7. ANNOTATION LOOP
# ============================================================

processed_since_checkpoint = 0
consecutive_errors = 0

try:

    for i in tqdm(pending, desc="GPT-6.1 Sol — Full dataset"):

        target = df_model.loc[i, TEXT_COLUMN]

        if pd.isna(target) or str(target).strip() == "":
            print(f"Skipping empty sentence at row {i}")
            continue

        start = time.perf_counter()

        try:

            result = code_self_other_valence(
                target_sentence=target,
                country=df_model.loc[i, COUNTRY_COLUMN],
                referent_category=df_model.loc[i, REFERENT_COLUMN],
                previous_sentence=df_model.loc[i, PREVIOUS_COLUMN],
                next_sentence=df_model.loc[i, NEXT_COLUMN],
                model=MODEL,
                max_retries=MAX_RETRIES
            )

        except Exception as e:

            result = {
                "self_other": None,
                "valence": None,
                "error": f"{type(e).__name__}: {e}"
            }

        elapsed = time.perf_counter() - start

        # Prepare a durable record before changing the dataframe
        record = {
            "model": MODEL,
            "row_index": int(i),
            "prompt_sha256": df_model.loc[
                i, "annotation_prompt_sha256"
            ],
            "self_other_api_new": result.get("self_other"),
            "valence_api_new": result.get("valence"),
            "api_error": result.get("error"),
            "input_tokens": result.get("input_tokens"),
            "output_tokens": result.get("output_tokens"),
            "total_tokens": result.get("total_tokens"),
            "cached_input_tokens": result.get("cached_input_tokens"),
            "elapsed_seconds": float(elapsed)
        }

        # ---------------------------------------------
        # A. Write recovery log after every API request
        # ---------------------------------------------

        # Stop if Drive cannot record the new annotation.
        # This prevents continuing with unsaved results.

        with open(RECOVERY_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

        # ---------------------------------------------
        # B. Update the main dataframe
        # ---------------------------------------------

        for col in OUTPUT_COLUMNS:
            df_model.loc[i, col] = record.get(col)

        processed_since_checkpoint += 1

        success = (
            result.get("self_other") in LABELS
            and result.get("valence") in LABELS
        )

        if success:

            consecutive_errors = 0

        else:

            consecutive_errors += 1

            print(
                f"\nError in row {i}: "
                f"{result.get('error')}"
            )

        # ---------------------------------------------
        # C. Save full CSV every 200 observations
        # ---------------------------------------------

        if processed_since_checkpoint >= CHECKPOINT_EVERY:

            save_full_checkpoint()

            processed_since_checkpoint = 0

        # ---------------------------------------------
        # D. Stop after repeated errors
        # ---------------------------------------------

        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:

            print(
                "\nStopping after three consecutive failures. "
                "Progress has been recorded."
            )

            break

except KeyboardInterrupt:

    print("\nRun interrupted by user.")

except Exception as e:

    print(
        f"\nRun stopped: {type(e).__name__}: {e}"
    )

finally:

    # Best-effort final save.
    # Recovery log remains available if this save fails.
    try:
        save_full_checkpoint()
    except Exception as e:
        print(
            "Final CSV save failed. "
            f"Recovery log may contain additional progress: {e}"
        )


# ============================================================
# 8. FINAL PROGRESS REPORT
# ============================================================

completed_final = (
    pd.to_numeric(
        df_model["self_other_api_new"],
        errors="coerce"
    ).isin(LABELS)
    &
    pd.to_numeric(
        df_model["valence_api_new"],
        errors="coerce"
    ).isin(LABELS)
)

elapsed_values = pd.to_numeric(
    df_model["elapsed_seconds"],
    errors="coerce"
)

print("\n=========================================")
print("FINAL ANNOTATION STATUS")
print("=========================================")

print("Model:", MODEL)
print("Total sentences:", len(df_model))
print("Completed:", int(completed_final.sum()))
print("Still missing:", int((~completed_final).sum()))

print(
    "Recorded processing time (hours):",
    round(elapsed_values.sum() / 3600, 2)
)

print("\nMain CSV:", OUTPUT_FILE)
print("Recovery log:", RECOVERY_FILE)
print("=========================================")

Total sentences: 11535
Existing checkpoint verified.

Checking recovery log...
Annotations recovered from log: 0

GPT-6.1 SOL — FULL DATASET
Total observations: 11535
Already completed: 6070
Remaining: 5465
Requests planned this session: 5465
Test mode: False

Full CSV checkpoint saved: west_self_other_valence_gpt-6_1-sol_full_dataset_medium.csv


GPT-6.1 Sol — Full dataset:   4%|▎         | 200/5465 [10:20<5:21:23,  3.66s/it]

Full CSV checkpoint saved: west_self_other_valence_gpt-6_1-sol_full_dataset_medium.csv


GPT-6.1 Sol — Full dataset:   7%|▋         | 400/5465 [20:42<5:53:06,  4.18s/it]

Full CSV checkpoint saved: west_self_other_valence_gpt-6_1-sol_full_dataset_medium.csv


GPT-6.1 Sol — Full dataset:  11%|█         | 600/5465 [30:58<4:55:55,  3.65s/it]

... (47 outputs left out)

GPT-6.1 Sol — Full dataset:  99%|█████████▉| 5400/5465 [4:36:45<03:23,  3.14s/it]

Full CSV checkpoint saved: west_self_other_valence_gpt-6_1-sol_full_dataset_medium.csv


GPT-6.1 Sol — Full dataset: 100%|██████████| 5465/5465 [4:40:15<00:00,  3.08s/it]


Full CSV checkpoint saved: west_self_other_valence_gpt-6_1-sol_full_dataset_medium.csv

FINAL ANNOTATION STATUS
Model: gpt-6.1-sol
Total sentences: 11535
Completed: 11535
Still missing: 0
Recorded processing time (hours): 10.04

Main CSV: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-6_1-sol_full_dataset_medium.csv
Recovery log: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-6_1-sol_full_dataset_medium_recovery.jsonl
